# Time-Series Metrics with RedisTimeSeries

This notebook demonstrates using `AgentMetrics` for built-in observability.

**Features:**
- Record agent request latency
- Track token counts (input/output)
- Monitor cache hit rates
- Query aggregated statistics

**Metrics Collected:**
- `latency_ms` - Request processing time
- `input_tokens` - Input token count
- `output_tokens` - Output token count
- `cache_hit` - Cache hit indicator (0/1)

## Setup

In [1]:
import os
import time
import random
from dotenv import load_dotenv

load_dotenv()

REDIS_URL = os.getenv("REDIS_URL", "redis://redis:6379")
print(f"Redis URL: {REDIS_URL}")

Redis URL: redis://redis:6379


## Create Metrics Collector

In [2]:
from redis_openai_agents import AgentMetrics

# Create metrics collector
metrics = AgentMetrics(
    name="demo_agent",
    redis_url=REDIS_URL,
)

print(f"Metrics collector created: {metrics.name}")

Metrics collector created: demo_agent


## Record Metrics

In [3]:
# Simulate recording metrics from agent requests
print("Recording sample metrics...\n")

for i in range(10):
    # Simulate variable latency and token counts
    latency = random.uniform(50, 500)
    input_tokens = random.randint(10, 200)
    output_tokens = random.randint(20, 400)
    cache_hit = random.choice([True, False])
    
    metrics.record(
        latency_ms=latency,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        cache_hit=cache_hit,
    )
    
    print(f"Request {i+1}: latency={latency:.1f}ms, tokens={input_tokens}/{output_tokens}, cache_hit={cache_hit}")
    time.sleep(0.1)  # Small delay between samples

print("\nMetrics recorded!")

Recording sample metrics...

Request 1: latency=406.0ms, tokens=60/339, cache_hit=False
Request 2: latency=81.1ms, tokens=116/230, cache_hit=False


Request 3: latency=208.2ms, tokens=142/181, cache_hit=False
Request 4: latency=188.4ms, tokens=12/285, cache_hit=False


Request 5: latency=89.9ms, tokens=166/104, cache_hit=False
Request 6: latency=425.3ms, tokens=114/181, cache_hit=True


Request 7: latency=109.7ms, tokens=158/24, cache_hit=False
Request 8: latency=407.8ms, tokens=66/156, cache_hit=False


Request 9: latency=146.2ms, tokens=74/107, cache_hit=True
Request 10: latency=134.1ms, tokens=13/232, cache_hit=True



Metrics recorded!


## Query Statistics

In [4]:
# Get aggregated statistics
stats = metrics.get_stats()

print("Agent Metrics Summary")
print("=" * 40)
print(f"Total Requests: {stats['count']}")
print(f"\nLatency (ms):")
print(f"  Average: {stats['latency_avg']:.2f}")
print(f"  Min: {stats['latency_min']:.2f}")
print(f"  Max: {stats['latency_max']:.2f}")
print(f"\nTokens:")
print(f"  Total Input: {stats['input_tokens_sum']:.0f}")
print(f"  Total Output: {stats['output_tokens_sum']:.0f}")
print(f"\nCache:")
print(f"  Hit Rate: {stats['cache_hit_rate']:.1%}")

Agent Metrics Summary
Total Requests: 10

Latency (ms):
  Average: 219.66
  Min: 81.10
  Max: 425.28

Tokens:
  Total Input: 921
  Total Output: 1839

Cache:
  Hit Rate: 30.0%


## Query Specific Time Range

In [5]:
# Query latency over last minute
from_time = int((time.time() - 60) * 1000)  # Last 60 seconds
to_time = int(time.time() * 1000)

latency_data = metrics.range(
    metric="latency",
    from_time=from_time,
    to_time=to_time,
)

print(f"Latency samples in last 60 seconds: {len(latency_data)}")
for ts, value in latency_data[:5]:  # Show first 5
    print(f"  {ts}: {value:.2f}ms")

Latency samples in last 60 seconds: 10
  1768430119671: 405.97ms
  1768430119774: 81.10ms
  1768430119876: 208.16ms
  1768430119977: 188.36ms
  1768430120079: 89.93ms


## Cleanup

In [6]:
metrics.delete()
print("Metrics data deleted")

Metrics data deleted


## Summary

This notebook demonstrated:

1. **Metrics Collection** - Record latency, tokens, and cache hits
2. **Statistics Query** - Get aggregated stats (avg, min, max, sum)
3. **Time Range Query** - Query metrics over specific time windows

**Use Cases:**
- Monitor agent performance in production
- Track token usage and costs
- Measure cache effectiveness
- Alert on latency anomalies